<a href="https://colab.research.google.com/github/giuliabugatti09/PPE-Detector-CNN-/blob/main/cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Project: PPE Detection using Convolutional Neural Networks (CNN)
## Goal: Build a classification model from scratch to detect "Helmet" vs "Head".



## 1. Introduction & Environment Setup

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import zipfile
from sklearn.model_selection import train_test_split

# Keras Components
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# --- DATASET UNZIPPING ---
dataset_dir = "dataset"
zip_filename = "Hard Hat Workers.v2-raw.multiclass.zip"

if os.path.exists(zip_filename):
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    print(f"✅ Success: Dataset extracted to '{dataset_dir}'")
else:
    print(f"⚠️ Warning: {zip_filename} not found. Ensure the zip is in the root folder.")

✅ Success: Dataset extracted to 'dataset'


## 2. Data Preparation (CSV to Flow)

In [2]:
# Path to the training metadata
train_csv_path = os.path.join(dataset_dir, "train", "_classes.csv")
train_img_dir = os.path.join(dataset_dir, "train")

# Load and clean the CSV
train_df = pd.read_csv(train_csv_path)
train_df.columns = [c.strip() for c in train_df.columns] # Remove white spaces

# Conversion for Binary Classification:
# We need to map the one-hot columns back to a single string label for flow_from_dataframe
def get_label(row):
    if row['helmet'] == 1: return 'helmet'
    return 'head'

train_df['label'] = train_df.apply(get_label, axis=1)

# Display first few rows to confirm
print("--- Training Data Sample ---")
display(train_df[['filename', 'label']].head())

--- Training Data Sample ---


,filename,label
0,003626_jpg.rf.0024e8fc3c8c3f411962ca8dab7b8e92...,helmet
1,004434_jpg.rf.002a70f061745a217db4320ae7b402a7...,head
2,004858_jpg.rf.002ab521984d81c7400faa6f916f5a01...,head
3,002310_jpg.rf.0008cd4590d2edb0e1447329236d9c11...,helmet
4,004785_jpg.rf.002ffb29898b3cba483a76e8b73d91a8...,helmet


## 3. Data Augmentation & Generators

In [3]:
# Parameters
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# Training Generator with Strong Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2 # Automatically splits 20% for validation
)

# Train Generator
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_img_dir,
    x_col="filename",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training",
    shuffle=True
)

# Validation Generator
val_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_img_dir,
    x_col="filename",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    shuffle=False
)

Found 4216 validated image filenames belonging to 2 classes.
Found 1053 validated image filenames belonging to 2 classes.


## 4. CNN Model Architecture (From Scratch)

In [4]:
model = Sequential([
    Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name="input_layer"),

    # Block 1
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Block 2
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Block 3
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Classification Head
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Sigmoid for binary classification
])

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,785 (432.75 KB)

 Trainable params: 110,337 (431.00 KB)

 Non-trainable params: 448 (1.75 KB)

## 5. Training & Evaluation

In [5]:
checkpoint = ModelCheckpoint(
    'best_epi_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max'
)

# Training
history = model.fit(
    train_generator,
    epochs=50,
    validation_data=val_generator,
    callbacks=[checkpoint]
)

# Final Save
model.save("final_ppe_model.keras")

Epoch 1/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 303s 2s/step - accuracy: 0.9051 - loss: 0.3127 - val_accuracy: 0.9278 - val_loss: 0.5443
Epoch 2/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 280s 2s/step - accuracy: 0.9137 - loss: 0.2649 - val_accuracy: 0.9278 - val_loss: 0.3887
Epoch 3/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 294s 2s/step - accuracy: 0.9115 - loss: 0.2521 - val_accuracy: 0.9278 - val_loss: 0.2992
Epoch 4/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 286s 2s/step - accuracy: 0.9137 - loss: 0.2404 - val_accuracy: 0.9307 - val_loss: 0.2823
Epoch 5/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 292s 2s/step - accuracy: 0.9146 - loss: 0.2397 - val_accuracy: 0.9297 - val_loss: 0.2175
Epoch 6/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 282s 2s/step - accuracy: 0.9156 - loss: 0.2330 - val_accuracy: 0.9278 - val_loss: 0.2347
Epoch 7/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 295s 2s/step - accuracy: 0.9175 - loss: 0.2276 - val_accuracy: 0.9297 - val_loss: 0.1866
Epoch 8/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 278s 2s/step - accuracy: 0.9151 - loss: 0.2279 - val_accu

## 6. Deployment-Ready Artifacts

In [6]:
# To download from Colab
try:
    from google.colab import files
    files.download('best_epi_model.keras')
except:
    print("Running locally. Model saved in the root directory.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>